# Agent Evaluation Lab

This notebook runs one agent at a time or the full orchestrator against CSV-driven test cases, then scores hallucination risk, tool-use accuracy, reliability, and failure patterns.

Before running a batch:
- edit the placeholder CSVs in `../data/evals/`
- duplicate placeholder rows instead of overwriting your whole template
- set `enabled=true` only for rows you want to execute
- fill only the expectation columns you actually trust

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.evaluation.notebook_lab import (
    TARGET_TO_CASE_FILE,
    assertions_dataframe,
    load_case_rows,
    results_dataframe,
    run_case_batch,
    save_results_csv,
    summarize_results,
)
from src.utils.config import Settings

sns.set_theme(style="darkgrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [ ]:
TARGET = "classification"  # classification | landing | hosting | embedded | orchestrator
ACTIVE_ONLY = True
LIMIT = None
PERSIST_ORCHESTRATOR_RUNS = False

settings = Settings.from_yaml()
case_path = REPO_ROOT / TARGET_TO_CASE_FILE[TARGET]
case_rows = load_case_rows(case_path)

print(f"Target: {TARGET}")
print(f"CSV: {case_path}")
print(f"Rows in file: {len(case_rows)}")


In [ ]:
case_df = pd.DataFrame(case_rows)
display(case_df)

active_case_df = case_df[case_df["enabled"] == True] if ACTIVE_ONLY else case_df.copy()
print(f"Rows selected for execution: {len(active_case_df)}")


In [ ]:
results = await run_case_batch(
    TARGET,
    csv_path=case_path,
    settings=settings,
    enabled_only=ACTIVE_ONLY,
    limit=LIMIT,
    persist_orchestrator_runs=PERSIST_ORCHESTRATOR_RUNS,
)

results_df = results_dataframe(results)
display(results_df)


In [ ]:
summary = summarize_results(results)
summary_df = pd.DataFrame([summary["overall"]])
failure_modes_df = pd.DataFrame([
    {"failure_mode": key, "count": value}
    for key, value in summary["failure_modes"].items()
])

display(summary_df)
display(failure_modes_df.sort_values("count", ascending=False) if not failure_modes_df.empty else failure_modes_df)


In [ ]:
assertions_df = assertions_dataframe(results)
display(assertions_df)

failing_assertions_df = assertions_df[assertions_df["passed"] == False] if not assertions_df.empty else assertions_df
display(failing_assertions_df)


In [ ]:
if not results_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.barplot(data=results_df, x="case_id", y="hallucination_score", ax=axes[0], color="#16b8a6")
    sns.barplot(data=results_df, x="case_id", y="tool_accuracy_score", ax=axes[1], color="#ff8c42")
    sns.barplot(data=results_df, x="case_id", y="reliability_score", ax=axes[2], color="#75a9ff")
    axes[0].set_title("Hallucination Score")
    axes[1].set_title("Tool Accuracy")
    axes[2].set_title("Reliability")
    for ax in axes:
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

    latency_plot_df = results_df[["case_id", "latency_ms", "total_cost_usd"]].copy()
    display(latency_plot_df.sort_values("latency_ms", ascending=False))


In [ ]:
report_dir = REPO_ROOT / "data" / "reports" / "notebook_lab" / TARGET
report_dir.mkdir(parents=True, exist_ok=True)

results_csv_path = save_results_csv(results, report_dir / "latest_results.csv")
assertions_csv_path = report_dir / "latest_assertions.csv"
assertions_df.to_csv(assertions_csv_path, index=False)

print(f"Saved results to: {results_csv_path}")
print(f"Saved assertions to: {assertions_csv_path}")


## Suggested Next Steps

- Start with the single-agent CSVs to stabilize classification and extraction assumptions.
- Once those pass consistently, move the strongest sites into `orchestrator_cases.csv`.
- Use `expected_provider_keywords`, `expected_stream_host_keywords`, and `expected_failure_mode` generously. They make the failure analysis much more useful than raw pass/fail alone.
- Keep some intentionally hard or flaky cases in the dataset. They help surface reliability regressions earlier.